# ColPali Multimodal Search with GEM Native Index

This notebook demonstrates voyager-index's multimodal search capabilities
using the ColPali vision-language model architecture and GEM native index.

**ColPali** encodes document pages (images/PDFs) into per-patch embeddings,
enabling visual document understanding with late-interaction retrieval.

**What you'll learn:**
- Index visual documents as multi-vector embeddings
- Text-to-image search (find images by description)
- Cross-modal similarity
- MaxSim scoring mechanics
- Scaling behavior at different dataset sizes

In [ ]:
import voyager_index
import numpy as np
import time
import tempfile

print(f"voyager-index version: {voyager_index.__version__}")

## 1. Simulating ColPali Patch Embeddings

In production, ColPali outputs a grid of patch embeddings per image.
A 224x224 image with 16x16 patches produces 196 patch vectors.
We simulate this with topically clustered embeddings.

In [ ]:
DIM = 128
N_IMAGES = 1000
PATCHES_PER_IMAGE = 196  # 14x14 grid from ViT

rng = np.random.RandomState(123)
N_VISUAL_TOPICS = 8
topic_centers = rng.randn(N_VISUAL_TOPICS, DIM).astype(np.float32) * 2.0

topic_names = [
    "landscape", "portrait", "document", "chart",
    "architecture", "food", "medical", "satellite",
]

images = []
payloads = []
for i in range(N_IMAGES):
    topic = i % N_VISUAL_TOPICS
    # Each image has patches clustered around its topic center
    patches = topic_centers[topic] + rng.randn(PATCHES_PER_IMAGE, DIM).astype(np.float32) * 0.4
    images.append(patches)
    payloads.append({
        "type": topic_names[topic],
        "source": f"page_{i}.png",
        "n_patches": PATCHES_PER_IMAGE,
    })

print(f"Created {N_IMAGES} synthetic images")
print(f"Each image: {PATCHES_PER_IMAGE} patches x {DIM}d = {PATCHES_PER_IMAGE * DIM * 4 / 1024:.1f} KB")
print(f"Total raw: {N_IMAGES * PATCHES_PER_IMAGE * DIM * 4 / 1024**2:.1f} MB")

## 2. Create the Multimodal Index

In [ ]:
index_path = tempfile.mkdtemp(prefix="voyager_colpali_")

# Use IndexBuilder for fine-grained control
index = (voyager_index.IndexBuilder(index_path, dim=DIM)
         .with_gem(n_fine=128, n_coarse=16, max_degree=32,
                   ef_construction=100, seed_batch_size=128)
         .with_wal(enabled=True)
         .build())

print(f"Index created: {index}")

In [ ]:
t0 = time.perf_counter()
ids = index.add(images, payloads=payloads)
build_time = time.perf_counter() - t0

print(f"Indexed {len(ids)} images in {build_time:.1f}s")
print(f"Throughput: {len(ids) / build_time:.0f} images/s")
stats = index.stats()
print(f"Active vectors: {stats.active_vectors}")

## 3. Text-to-Image Search

Simulate a text query encoded by ColPali's text encoder into token vectors.

In [ ]:
# Simulate: "a mountain landscape with a lake"
query_topic = 0  # landscape
text_query = topic_centers[query_topic] + rng.randn(8, DIM).astype(np.float32) * 0.15

t0 = time.perf_counter()
results = index.search(text_query, k=10, ef=128)
search_ms = (time.perf_counter() - t0) * 1000

print(f"Text-to-Image search: {search_ms:.2f}ms\n")
print(f"{'Rank':<6} {'ID':<6} {'Score':<10} {'Type':<15} {'Source'}")
print("-" * 55)
for i, r in enumerate(results):
    print(f"{i+1:<6} {r.id:<6} {r.score:<10.4f} {r.payload['type']:<15} {r.payload['source']}")

# Verify results are mostly from the correct topic
correct = sum(1 for r in results if r.payload['type'] == 'landscape')
print(f"\nPrecision: {correct}/{len(results)} ({correct/len(results)*100:.0f}%) are landscapes")

## 4. Image-to-Image Search

Use an image's patch embeddings as the query to find visually similar images.

In [ ]:
# Use the first medical image's patches as query
query_img = images[6]  # topic 6 = "medical"

# Use a subset of patches as the query (like a crop)
query_patches = query_img[:32]  # first 32 patches

results = index.search(query_patches, k=10)

print("Image-to-Image search (medical image query):\n")
for i, r in enumerate(results):
    marker = " <-- query image" if r.id == 6 else ""
    print(f"  {i+1}. ID={r.id}, type={r.payload['type']}, score={r.score:.4f}{marker}")

## 5. MaxSim Scoring Explained

MaxSim computes: `score = sum_over_query_tokens(max_over_doc_patches(sim(q_i, d_j)))`

This means each query token finds its best matching patch, and scores are summed.

In [ ]:
# Manual MaxSim computation for understanding
query = text_query  # 8 query tokens
doc = images[0]     # 196 patches

# Similarity matrix: (n_query_tokens, n_doc_patches)
sim_matrix = query @ doc.T
print(f"Similarity matrix shape: {sim_matrix.shape}")
print(f"  (query tokens: {query.shape[0]}, doc patches: {doc.shape[0]})")

# MaxSim: for each query token, take the max similarity across patches
max_per_token = sim_matrix.max(axis=1)
print(f"\nMax similarity per query token: {max_per_token}")
print(f"MaxSim score (sum): {max_per_token.sum():.4f}")
print(f"\nThis is what GEM uses internally via quantized Chamfer (qCH) scoring,")
print(f"which approximates MaxSim with product quantization for sub-millisecond latency.")

## 6. Filtered Multimodal Search

In [ ]:
# Find charts matching a data-related query
query = topic_centers[3] + rng.randn(6, DIM).astype(np.float32) * 0.2  # chart topic

# Filter: only charts and documents
results = index.search(query, k=5, filters={"type": {"$in": ["chart", "document"]}})
print("Filtered search (charts + documents only):")
for r in results:
    print(f"  ID={r.id}, type={r.payload['type']}, score={r.score:.4f}")

print()

# Exclude medical and satellite imagery
results = index.search(query, k=5, filters={"type": {"$nin": ["medical", "satellite"]}})
print("Filtered search (excluding medical + satellite):")
for r in results:
    print(f"  ID={r.id}, type={r.payload['type']}, score={r.score:.4f}")

## 7. Scaling Performance

In [ ]:
n_queries = 50
latencies = []
for _ in range(n_queries):
    q = topic_centers[rng.randint(0, N_VISUAL_TOPICS)] + rng.randn(8, DIM).astype(np.float32) * 0.2
    t0 = time.perf_counter()
    _ = index.search(q, k=10)
    latencies.append((time.perf_counter() - t0) * 1000)

latencies.sort()
n = len(latencies)
print(f"Search latency over {n_queries} queries ({N_IMAGES} images, {PATCHES_PER_IMAGE} patches/image):")
print(f"  p50:  {latencies[n // 2]:.2f}ms")
print(f"  p95:  {latencies[int(n * 0.95)]:.2f}ms")
print(f"  p99:  {latencies[int(n * 0.99)]:.2f}ms")
print(f"  QPS:  {n / (sum(latencies) / 1000):.0f}")

In [ ]:
index.close()
print("Done! Index closed.")
print("\nKey takeaways:")
print("- ColPali produces per-patch embeddings (196 vectors per image)")
print("- GEM indexes these natively as multi-vector documents")
print("- Search uses quantized Chamfer (qCH) for sub-ms graph traversal")
print("- Payload filtering enables structured multimodal queries")
print("- The same Index API works for both ColBERT text and ColPali vision")